# Limpieza de Nombres - Customers
## Prueba la nueva funcion `clean_name` desde la arquitectura refactorizada

In [0]:
# Celda de configuración inicial
# Solo ejecútala PRIMERO, siempre

# Paso 1: Crear sesión Spark (en VS Code no viene automático como en Databricks web)
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.profile("ecommerce-etl-pipeline").getOrCreate()
    print("Sesión Spark creada via Databricks Connect")
except Exception as e:
    print(f"Usando spark existente: {e}")

# Paso 2: Agregar carpeta raíz del proyecto al path de Python
# (para que encuentre la carpeta transformacion/)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
print(f"Path de Python: {sys.path[0]}")

In [6]:
# 1. Cargar datos crudos de bronze
storage_account_name = "stecommercedirty01"
container = "bronze"

path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/customers/"
df_customers = spark.read.option("multiLine", True).json(path)

print(f"Filas: {df_customers.count()}")
df_customers.printSchema()

NameError: name 'spark' is not defined

In [3]:
# 2. Importar la nueva arquitectura de limpieza
from transformacion.cleaning import clean_name
from transformacion.models.customer import CLEANING_RULES, CUSTOMER_SCHEMA

print(f"Reglas de limpieza: {list(CLEANING_RULES.keys())}")

ModuleNotFoundError: No module named 'transformacion'

In [0]:
# 3. Ver los nombres sucios ANTES de limpiar
df_customers.select("first_name", "last_name", "email").show(20, truncate=False)

# Contar nulos y vacios
from pyspark.sql.functions import col, when, trim
nulos_fn = df_customers.filter(col("first_name").isNull() | (trim(col("first_name")) == "")).count()
print(f"First_name nulos/vacios: {nulos_fn}")
print(f"First_name no nulos: {df_customers.count() - nulos_fn}")

In [0]:
# 4. Aplicar clean_name a first_name, last_name y email
df_limpio = clean_name(df_customers, "first_name")
df_limpio = clean_name(df_limpio, "last_name")
df_limpio = clean_name(df_limpio, "email")

print("Listo! Columnas nuevas creadas:")
[c for c in df_limpio.columns if c.endswith("_clean")]

In [0]:
# 5. Comparar ANTES vs DESPUES de la limpieza
df_limpio.select(
    col("first_name"), col("first_name_clean"),
    col("last_name"), col("last_name_clean"),
    col("email"), col("email_clean")
).filter(col("first_name_clean") != col("first_name")) \
    .show(30, truncate=False)

In [0]:
# 6. Verificar caracteres especiales ELIMINADOS
from pyspark.sql.functions import regexp_extract

# Buscar nombres que AUN tengan caracteres especiales (malos)
especiales = df_limpio.filter(
    regexp_extract(col("first_name_clean"), "[^a-zA-Z\\s'-]", 0) != ""
).select("first_name", "first_name_clean")

print(f"Nombres con caracteres especiales restantes: {especiales.count()}")
especiales.show(10)

In [0]:
# 7. Verificar que los vacios ahora son UNKNOWN
df_limpio.filter(col("first_name_clean") == "UNKNOWN") \
    .select("first_name", "first_name_clean", "last_name_clean") \
    .show(20, truncate=False)

print(f"Total first_name -> UNKNOWN: {df_limpio.filter(col('first_name_clean') == 'UNKNOWN').count()}")

In [0]:
# 8. Vista final limpia
df_limpio.select(
    "customer_id", "first_name_clean", "last_name_clean", "email_clean"
).show(30, truncate=False)